In [ ]:
# ar/statistics-101/hard/16-exploratory-pass
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("students-performance.csv", ())


## ماذا يعني

**الجولة الاستكشافية** (exploratory pass) تسلسل ثابت من الفحوصات يُنفَّذ قبل استخلاص أي استنتاج. ومخرجاتها ليست نتيجة بل مجموعة قرارات مسجلة: ما المفقود، وما غير المعقول، وما المجمّع، وما الذي ستدّعيه بالتالي وما لن تدّعيه.

الانضباط هو فصل *النظر* عن *الاستنتاج*. النتائج الاستكشافية فرضيات؛ والنتائج التأكيدية تُختبر على بيانات لم تتشكل الفرضية عليها.

## الفحوصات الاثنا عشر

1. **شكل الجدول** — الصفوف والأعمدة وما يمثله الصف الواحد.
2. **الأنواع** — أي الأعمدة أرقام وصلت كنص.
3. **الفقدان** — العد لكل عمود، وهل يتكتل حسب المجموعة.
4. **المديات** — الأدنى والأعلى لكل عمود رقمي، بحثًا عن قيم مستحيلة.
5. **التكرارات** — صفوف متطابقة، ومعرّفات متكررة.
6. **المركز** — المتوسط والوسيط جنبًا إلى جنب.
7. **التشتت** — الانحراف المعياري والمدى الربيعي جنبًا إلى جنب.
8. **الذيل** — كم قيمة تقع بعد الأسوار، وما هي.
9. **المجموعات** — هل تختلف الملخصات حسب الفئة أو الموقع أو الدفعة؟
10. **العلاقات** — مخططات الانتشار قبل أي رقم ارتباط.
11. **القرارات** — كل استبعاد، مع سبب مسجل.
12. **الادعاءات** — فقرة عما تدعمه البيانات، وفقرة عما لا تدعمه.

## مثال محسوب

يحزم المقرر مجموعة بيانات أداء الطلاب. نمّطها أولًا:


In [ ]:
import pandas as pd

df = pd.read_csv('students-performance.csv')
print(df.shape)          # how many rows, columns
print(df.dtypes)         # check that scores are numbers
print(df.isna().sum())   # missing per column
print(df.describe())     # ranges, centre, spread in one view


**ما يخبرك به الناتج، بالترتيب.**

- إذا أظهر `shape` صفوفًا أكثر مما تتوقع، فافحص التكرارات قبل المضي.
- إذا كان لعمود الدرجات نوع `object`، فهو يحتوي قيمة شاذة مثل `"absent"`، وكل عملية حسابية بعد ذلك ستكون خاطئة أو متجاهلة بصمت.
- إذا كان `isna().sum()` غير صفري، فقرر الآن هل تحذف أو تملأ أو تنمذج الفقدان؛ ويجب تسجيل الخيار.
- إذا أظهر `describe()` أدنى قيمة 0 في عمود درجات، فقد يكون ذلك علامة غياب لا صفرًا حقيقيًا.

ثم افحص شكل درجة واحدة:


In [ ]:
score = df['math score']
print(score.mean(), score.median())
q1, q3 = score.quantile([.25, .75])
print(q1, q3, q3 - q1)
print((score < q1 - 1.5*(q3-q1)).sum(), (score > q3 + 1.5*(q3-q1)).sum())


**قراءته.** إذا اختلف المتوسط والوسيط بنقطة أو نقطتين فقط، فالتوزيع قريب من التماثل وأي مركز مقبول. وإذا كان الانحراف المعياري أكبر بكثير من المدى الربيعي مقسومًا على 1.35، فالأذيال أثقل من الشكل الطبيعي والمتوسط مسحوب. وإذا كانت أعداد الأسوار غير صفرية، فافحص تلك الصفوف فردًا فردًا.

<details class="challenge">
<summary>🧩 تحدٍّ، فكّر أولًا ثم أظهر</summary>
<div class="challenge__body">

يُظهر `describe()` أدنى مدة لدورة تحضير للاختبار 0 وأعلى مدة 100، بمتوسط 35. هل يبرر ذلك معاملة 0 كـ"لا تحضير" أم كبيانات مفقودة؟ ماذا تنظر إليه؟

<p class="challenge__answer">💡 <strong>الجواب:</strong> التوزيع هو الحكم. إذا كان العمود متصلًا في بقية قيمه وتقع نسبة كبيرة عند 0 بالضبط، فهو غالبًا فئة "لا شيء" حقيقية أو قيمة افتراضية؛ وإذا كانت حفنة فقط عند 0 بينما بقية القيم متوزعة، فالأرجح أنها قيم مفقودة مرمزة بـ0. جدول الصفرات مقابل متغير آخر (مثل تعليم الوالدين، الذي يجب ألا يرتبط بالفقدان) للتحقق مما إذا كانت تتصرف كملاحظات حقيقية.</p>

</div>
</details>

## تسجيل القرارات

مخرج هذا الدرس سجل قصير، سطر لكل قرار:


In [ ]:
حُذفت 3 صفوف: معرّفات طلاب مكررة (2) ودرجة واحدة مقدارها 1000 (إدخال بيانات).
أُبقيت 12 صفًا بقيمة تحضير للاختبار = 0: التوزيع متصل، ولا عمود آخر مفقود لها.
جُمّعت حسب التحضير للاختبار: 'لا شيء' و'مكتمل' يختلفان بـ8 نقاط في القراءة والكتابة.


هذا السجل هو ما يجعل التحليل قابلًا للمراجعة. دونه، لا يستطيع القارئ التمييز بين استبعاد مدروس واستبعاد مريح.

## متى تستخدمه

قبل أي تحليل تأكيدي، وقبل تسليم البيانات لأي شخص آخر. إنها رخيصة — دقائق — بالمقارنة بتكلفة استنتاج مبني على عمود أُسيء قراءته.

## القيود

- **النتائج الاستكشافية ليست دليلًا.** الجولة نفسها التي تجد نمطًا تولّد الفرضية أيضًا، لذا يجب فحص النمط على بيانات جديدة.
- **تتوسع بشكل سيئ دون تجميع.** مع آلاف الفئات تحتاج الفحوصات الاثنا عشر إلى استراتيجية معاينة لا إلى سرد كامل.
- **لا تستطيع كشف ما ليس في البيانات.** المجموعات الغائبة والإخفاقات غير المسجلة وتحيز البقاء غير مرئية لتنميط ما جُمع.

## أخطاء شائعة

- **الإكراه الصامت.** تحويل النص إلى أرقام بـ`pd.to_numeric(errors='coerce')` يحوّل القيم السيئة إلى مفقودة، فيقفز عدد الفقدان؛ ويجب تسجيل الاثنين.
- **الاكتفاء بـ`describe()` بدلًا من مخطط.** إنه يخفي القمم المتعددة تمامًا.
- **حذف الذيل قبل التحقق مما إذا كان مجموعة فرعية حقيقية.**

## 🧩 تحديات

<details class="challenge">
<summary>🧩 تحدٍّ، فكّر أولًا ثم أظهر</summary>
<div class="challenge__body">

يظهر في تنميطك قمتان في المدرج التكراري للدرجات. ما التفسيران الأرجح، وما الذي يميز بينهما؟

<p class="challenge__answer">💡 <strong>الجواب:</strong> إما أن مجموعة البيانات تحتوي مجتمعين مختلفين (مثلًا مدرستان، أو تطبيقان للاختبار)، أو أن المقياس مقصوص أو مقرّب بطريقة تكدس القيم. ميّز بينهما بتلوين المدرج التكراري بمتغير تجميع مرشح؛ فإذا انفصلت القمتان بوضوح، فالمجتمعان مختلفان، وكل ملخص مدمج يجب تقسيمه حسب ذلك المتغير.</p>

</div>
</details>

## 🤔 أسئلة سقراطية

- لماذا يهم تسجيل القرارات لا نتائجها فقط؟
- أي الفحوصات الاثني عشر سيكشف مفارقة سيمبسون قبل أن تبلغ أي شيء؟
- إذا كان عمود مفقودًا بنسبة 30%، فما ثلاث طرق مشروعة للمضي، وماذا يفترض كل منها؟
- لماذا تكون النتيجة الاستكشافية دليلًا أضعف من التأكيدية، حتى حين تتطابق الأرقام؟

## ✅ فحص سريع

<div class="quiz" data-quiz="statistics-101-eda-ar">
  <div class="quiz-q" data-answer="2">
    <p class="quiz-q__prompt">1. يصل عمود رقمي بنوع object. ما أول شيء تفحصه؟</p>
    <div class="quiz-q__options">
      <button class="quiz-q__opt" data-idx="0">المتوسط.</button>
      <button class="quiz-q__opt" data-idx="1">الارتباط مع الأعمدة الأخرى.</button>
      <button class="quiz-q__opt" data-idx="2">أي الإدخالات ليست أرقامًا قابلة للتحليل، وماذا تمثل.</button>
      <button class="quiz-q__opt" data-idx="3">هل تحذف العمود.</button>
    </div>
    <p class="quiz-q__feedback" hidden></p>
  </div>
  <div class="quiz-q" data-answer="3">
    <p class="quiz-q__prompt">2. لماذا تسجل قرار استبعاد؟</p>
    <div class="quiz-q__options">
      <button class="quiz-q__opt" data-idx="0">لجعل التقرير أطول.</button>
      <button class="quiz-q__opt" data-idx="1">لأن البرمجية تطلب ذلك.</button>
      <button class="quiz-q__opt" data-idx="2">لتسريع التحليل التالي.</button>
      <button class="quiz-q__opt" data-idx="3">ليتمكن القارئ من التمييز بين خيار مدروس وخيار مريح، ومن إعادة إنتاجه.</button>
    </div>
    <p class="quiz-q__feedback" hidden></p>
  </div>
  <p class="quiz__summary" data-quiz-summary hidden></p>
</div>


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
